# Bike Sharing Data: EDA and Handling Missing Values
This notebook demonstrates basic Exploratory Data Analysis (EDA) and applies specific strategies for handling missing values in the `bikes.csv` dataset.

## Step 1: Import Libraries
First, we import the necessary Python libraries for data manipulation and imputation.

In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

## Step 2: Load Data & Inspect Structure
We will load the dataset and take a look at the first few rows, the overall shape of the data (number of rows and columns), and the data types of each feature.

In [5]:
# Load the dataset
df = pd.read_csv('bikes.csv')
# Display the first 5 rows to understand what the data looks like
print("--- First 5 Rows of the Dataset ---")
display(df.head())


--- First 5 Rows of the Dataset ---


,datetime,season,holiday,workingday,weather,temp,humidity,windspeed,casual,registered,rented_bikes_count
0,2011-01-01 00:00:00,Spring,0.0,0.0,Clear,9.84,81.0,NaN,3,13,16
1,2011-01-01 01:00:00,Spring,0.0,0.0,NaN,9.02,80.0,0.0,8,32,40
2,2011-01-01 02:00:00,Spring,0.0,0.0,Clear,9.02,NaN,0.0,5,27,32
3,2011-01-01 03:00:00,Spring,0.0,0.0,Clear,9.84,75.0,0.0,3,10,13
4,2011-01-01 04:00:00,NaN,0.0,0.0,Clear,NaN,75.0,NaN,0,1,1


In [ ]:
# Display data types and non-null counts and shape
print("--- Data Types and Basic Info ---")
df.info()


Dataset Shape: 10886 rows and 11 columns.

--- Data Types and Basic Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   datetime            10886 non-null  object 
 1   season              10672 non-null  object 
 2   holiday             10030 non-null  float64
 3   workingday          9388 non-null   float64
 4   weather             8746 non-null   object 
 5   temp                8104 non-null   float64
 6   humidity            7462 non-null   float64
 7   windspeed           6820 non-null   float64
 8   casual              10886 non-null  int64  
 9   registered          10886 non-null  int64  
 10  rented_bikes_count  10886 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 935.6+ KB


## Step 3: Summary Statistics & Duplicates Check
Next, we will generate a statistical summary of the numerical and categorical features to understand their distribution, and check if there are any duplicate rows in our dataset.

In [8]:
# Statistical summary of numerical columns (mean, min, max, quartiles)
print("--- Summary Statistics (Numerical) ---")
display(df.describe())

# Statistical summary of categorical columns (count, unique, top, freq)
print("--- Summary Statistics (Categorical) ---")
display(df.describe(include='object'))

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

# Remove duplicates if any exist
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates have been successfully removed.")

--- Summary Statistics (Numerical) ---


,holiday,workingday,temp,humidity,windspeed,casual,registered,rented_bikes_count
count,10030.000000,9388.000000,8104.000000,7462.000000,6820.000000,10886.000000,10886.000000,10886.000000
mean,0.029113,0.678206,20.317665,61.790673,12.708806,36.021955,155.552177,191.574132
std,0.168131,0.467189,7.818568,19.262084,8.131154,49.960477,151.039033,181.144454
min,0.000000,0.000000,0.820000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,13.940000,47.000000,7.001500,4.000000,36.000000,42.000000
50%,0.000000,1.000000,20.500000,62.000000,12.998000,17.000000,118.000000,145.000000
75%,0.000000,1.000000,26.240000,77.000000,16.997900,49.000000,222.000000,284.000000
max,1.000000,1.000000,41.000000,100.000000,56.996900,367.000000,886.000000,977.000000


--- Summary Statistics (Categorical) ---


,datetime,season,weather
count,10886,10672,8746
unique,10886,4,4
top,2011-01-01 00:00:00,Winter,Clear
freq,1,2688,5793



Number of duplicate rows: 0


## Step 4: Missing Values Analysis
Before we apply our imputation strategies, we need to precisely quantify the missing values in each column to understand the scale of the problem.

In [13]:
# Count of missing values per column
print("--- Missing Values Count ---")
missing_values = df.isnull().sum()
display(missing_values[missing_values > 0]) # Display only columns that have missing values

# Percentage of missing values per column
print("\n--- Missing Values Percentage ---")
missing_percentage = (missing_values / len(df)) * 100
display(missing_percentage[missing_percentage > 0].round(2).astype(str) + ' %')

--- Missing Values Count ---


season         214
holiday        856
workingday    1498
weather       2140
temp          2782
humidity      3424
windspeed     4066
dtype: int64


--- Missing Values Percentage ---


season         1.97 %
holiday        7.86 %
workingday    13.76 %
weather       19.66 %
temp          25.56 %
humidity      31.45 %
windspeed     37.35 %
dtype: object

## Step 5: Handling Missing Values
Now that we have analyzed the missing data, we will apply the specific cleaning strategies required for each feature.

### 5.1 Drop Missing Values in 'season'
The `season` column is a core categorical variable. We will completely drop any rows where the season is missing to avoid bais and also becasue the percentge is less than 2%.


In [14]:
df.dropna(subset=['season'], inplace=True)
print(f"Dataset shape after dropping missing seasons: {df.shape}")

Dataset shape after dropping missing seasons: (10672, 11)


### 5.2 Impute Missing Values with Mode
For the categorical columns `holiday`, `workingday`, and `weather`, we will fill the missing values with their most frequent occurrence (Mode).

In [20]:
df['workingday'].mode()

0    1.0
Name: workingday, dtype: float64

In [23]:
for col in ['holiday','workingday','weather']:
    mode_value=df[col].mode()[0] # to take only mode value as the result is series
    df[col].fillna(mode_value,inplace=True)
    print(f'We filled Missing values in "{col}"" with its mode :{mode_value}')


We filled Missing values in "holiday"" with its mode :0.0
We filled Missing values in "workingday"" with its mode :1.0
We filled Missing values in "weather"" with its mode :Clear


### 5.3 Impute Missing Values with Median
For the continuous variable `humidity`, we will use the median to handle missing values, as it is robust to potential outliers.

In [25]:
median_value = df['humidity'].median()
df['humidity'].fillna(median_value, inplace=True)
print(f"Filled missing values in 'humidity' with its median: {median_value}")

Filled missing values in 'humidity' with its median: 62.0


### 5.4 Impute Missing Values using KNN Imputer
For environmental continuous variables like `temp` and `windspeed`, we use a K-Nearest Neighbors (KNN) imputer. This method estimates the missing values based on the similarity of other data points.

In [ ]:
# Initialize KNNImputer with 5 neighbors
knn_imputer = KNNImputer(n_neighbors=5)

# Apply KNN imputation to 'temp' and 'windspeed'
df[['temp', 'windspeed']] = knn_imputer.fit_transform(df[['temp', 'windspeed']])

Successfully applied KNN Imputation to 'temp' and 'windspeed'.


In [27]:
# Final check for missing values
print("--- Missing Values After Full Cleaning ---")
final_missing = df.isnull().sum()
print(final_missing)

--- Missing Values After Full Cleaning ---
datetime              0
season                0
holiday               0
workingday            0
weather               0
temp                  0
humidity              0
windspeed             0
casual                0
registered            0
rented_bikes_count    0
dtype: int64


In [ ]:
# Save the thoroughly cleaned dataset
cleaned_filename = 'cleaned_bikes_data.csv'
df.to_csv(cleaned_filename, index=False)
print(f"\nData successfully exported to '{cleaned_filename}'")